In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Next.js 14 introduces the App Router, shifting the paradigm towards React Server Components by default. This significantly reduces the client-side JavaScript bundle size and improves initial page load performance. For interactive UI elements, developers must explicitly use the 'use client' directive at the top of the file to opt into client-side rendering.",
        metadata={"source": "frontend_docs.md", "category": "web_development", "framework": "Next.js", "tier": "frontend"}
    ),
    Document(
        page_content="In Jetpack Compose, state hoisting is a pattern of moving state to a composable's caller to make a composable stateless. The general pattern for state hoisting in Compose is to replace the state variable with two parameters: a 'value' parameter representing the current state, and an 'onValueChange' lambda callback triggered when the UI requests an update.",
        metadata={"source": "android_ui_guidelines.md", "category": "mobile_development", "framework": "Jetpack Compose", "tier": "frontend"}
    ),
    Document(
        page_content="Hyperledger Fabric utilizes a deterministic consensus algorithm and requires smart contracts, known as chaincode, to be explicitly endorsed by specified peer nodes before a transaction is committed to the ledger. This custom endorsement policy is configured per channel and prevents malicious or non-deterministic code execution from corrupting the state.",
        metadata={"source": "blockchain_architecture.pdf", "category": "distributed_ledger", "framework": "Hyperledger", "tier": "backend"}
    ),
    Document(
        page_content="For the agricultural computer vision module, the convolutional neural network processes multispectral drone imagery to detect early signs of crop disease. The model identifies specific discoloration and lesion patterns on leaves, cross-referencing the bounding boxes with localized weather data pipelines to predict potential blight outbreaks.",
        metadata={"source": "vision_model_spec.txt", "category": "machine_learning", "project": "agritech_vision", "tier": "data_science"}
    ),
    Document(
        page_content="A modern real-time collaboration platform for coders requires either Operational Transformation (OT) or Conflict-free Replicated Data Types (CRDTs) to handle concurrent keystrokes safely. WebSockets maintain the persistent connection to the IDE client, while a Redis pub/sub mechanism distributes the real-time deltas across the distributed backend microservices.",
        metadata={"source": "system_design_doc.md", "category": "system_architecture", "project": "collaboration_platform", "tier": "infrastructure"}
    ),
    Document(
        page_content="In a DevOps CI/CD pipeline, Docker image caching can drastically reduce build times. By structuring the Dockerfile to install dependencies before copying the main application code, the build system can reuse the cached dependency layer as long as the requirements file remains unchanged.",
        metadata={"source": "devops_playbook.md", "category": "devops", "tool": "Docker", "tier": "infrastructure"}
    )
]

print(f"Loaded {len(documents)} documents into memory.")

Loaded 6 documents into memory.


In [2]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key= groq_api_key, model="llama-3.1-8b-instant")

c:\Users\acer\Desktop\Rag 2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")

In [4]:
from langchain_chroma import Chroma

vectorStore = Chroma.from_documents(documents , embedding=embeddings)
vectorStore

In [5]:
vectorStore.asimilarity_search("cat")

<coroutine object VectorStore.asimilarity_search at 0x000001DD45FB3970>

In [6]:
await vectorStore.asimilarity_search("cat")

[Document(id='29b1a2ec-1a56-4046-b1f3-1ad788835ca6', metadata={'category': 'machine_learning', 'project': 'agritech_vision', 'tier': 'data_science', 'source': 'vision_model_spec.txt'}, page_content='For the agricultural computer vision module, the convolutional neural network processes multispectral drone imagery to detect early signs of crop disease. The model identifies specific discoloration and lesion patterns on leaves, cross-referencing the bounding boxes with localized weather data pipelines to predict potential blight outbreaks.'),
 Document(id='b1c1d1fa-b764-4932-be7c-c7e8c2f72e02', metadata={'tier': 'frontend', 'source': 'android_ui_guidelines.md', 'category': 'mobile_development', 'framework': 'Jetpack Compose'}, page_content="In Jetpack Compose, state hoisting is a pattern of moving state to a composable's caller to make a composable stateless. The general pattern for state hoisting in Compose is to replace the state variable with two parameters: a 'value' parameter represe

In [7]:
vectorStore.similarity_search_with_score("cat")

[(Document(id='29b1a2ec-1a56-4046-b1f3-1ad788835ca6', metadata={'category': 'machine_learning', 'tier': 'data_science', 'project': 'agritech_vision', 'source': 'vision_model_spec.txt'}, page_content='For the agricultural computer vision module, the convolutional neural network processes multispectral drone imagery to detect early signs of crop disease. The model identifies specific discoloration and lesion patterns on leaves, cross-referencing the bounding boxes with localized weather data pipelines to predict potential blight outbreaks.'),
  1.731020212173462),
 (Document(id='b1c1d1fa-b764-4932-be7c-c7e8c2f72e02', metadata={'source': 'android_ui_guidelines.md', 'tier': 'frontend', 'category': 'mobile_development', 'framework': 'Jetpack Compose'}, page_content="In Jetpack Compose, state hoisting is a pattern of moving state to a composable's caller to make a composable stateless. The general pattern for state hoisting in Compose is to replace the state variable with two parameters: a '

### retreivers

In [8]:
from typing import List

from langchain_core.documents import Document 
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorStore.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])



[[Document(id='29b1a2ec-1a56-4046-b1f3-1ad788835ca6', metadata={'category': 'machine_learning', 'tier': 'data_science', 'source': 'vision_model_spec.txt', 'project': 'agritech_vision'}, page_content='For the agricultural computer vision module, the convolutional neural network processes multispectral drone imagery to detect early signs of crop disease. The model identifies specific discoloration and lesion patterns on leaves, cross-referencing the bounding boxes with localized weather data pipelines to predict potential blight outbreaks.')],
 [Document(id='b1c1d1fa-b764-4932-be7c-c7e8c2f72e02', metadata={'framework': 'Jetpack Compose', 'category': 'mobile_development', 'source': 'android_ui_guidelines.md', 'tier': 'frontend'}, page_content="In Jetpack Compose, state hoisting is a pattern of moving state to a composable's caller to make a composable stateless. The general pattern for state hoisting in Compose is to replace the state variable with two parameters: a 'value' parameter repr

In [9]:
retriever = vectorStore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='29b1a2ec-1a56-4046-b1f3-1ad788835ca6', metadata={'category': 'machine_learning', 'source': 'vision_model_spec.txt', 'project': 'agritech_vision', 'tier': 'data_science'}, page_content='For the agricultural computer vision module, the convolutional neural network processes multispectral drone imagery to detect early signs of crop disease. The model identifies specific discoloration and lesion patterns on leaves, cross-referencing the bounding boxes with localized weather data pipelines to predict potential blight outbreaks.')],
 [Document(id='b1c1d1fa-b764-4932-be7c-c7e8c2f72e02', metadata={'tier': 'frontend', 'category': 'mobile_development', 'source': 'android_ui_guidelines.md', 'framework': 'Jetpack Compose'}, page_content="In Jetpack Compose, state hoisting is a pattern of moving state to a composable's caller to make a composable stateless. The general pattern for state hoisting in Compose is to replace the state variable with two parameters: a 'value' parameter repr

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 1. Clean up the prompt template variables to be explicit
message = """
Answer the following question using the provided context only.

Question:
{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

# 2. Build the data-gathering step (RunnableParallel)
# When you invoke the chain with a string, RunnablePassthrough() grabs that string
# and the retriever uses that same string to fetch documents.
setup_and_retrieval = {
    "context": retriever, 
    "question": RunnablePassthrough()
}

# 3. Construct the actual pipeline (Left to Right)
# Gather Data -> Format Prompt -> Generate Answer
rag_chain = setup_and_retrieval | prompt | llm

# 4. Execute
# You pass the raw string. LCEL handles passing it to both the retriever and the passthrough.
response = rag_chain.invoke("Tell me about dogs")
print(response.content)

There is no information about dogs in the provided context. The context is about a convolutional neural network used in an agricultural computer vision module to detect early signs of crop disease.


: 